# Model run

In [ ]:
# Set scenario name
SCENARIO_NAME = 'current_hospitals'
NEW_MT_CENTRES = []

In [ ]:
import contextily as ctx
import geopandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from classes.geography_processing import Geoprocessing
from classes.model import Model
from classes.scenario import Scenario
from classes.add_net_benefit import add_net_benefit


In [ ]:
# Allow for a travel bias time (minutes) to MT centres
MT_travel_bias = 1

# Process and save geographic data (only needed when hospital data changes)
geo = Geoprocessing(mt_travel_bias = MT_travel_bias, new_mt_centres = NEW_MT_CENTRES)
geo.run()

# Set up scenario
scenario = Scenario({
    'name': SCENARIO_NAME,
    'limit_to_england' : True
})

# Set up model
model = Model(
    scenario=scenario,
    geodata=pd.read_csv('processed_data/processed_data.csv'))

# Run model
model.run()


In [ ]:
results = model.full_results

In [ ]:
results = model.full_results
# Add net benefit column
results = add_net_benefit(results, prop_lvo=0.3)
# Save results to CSV
results.to_csv(f'./output/results_full_{SCENARIO_NAME}.csv')

In [ ]:
results.head()

In [ ]:
# Get descriptive stats for all numeric columns
descriptive_stats = results.describe(percentiles=[0.25, 0.5, 0.75]).T
descriptive_stats.to_csv(f'./output/results_stats_{SCENARIO_NAME}.csv')
descriptive_stats

In [ ]:
# Repeat descriptive stats, grouped by nearest_ivt_unit
results_stats_by_unit = results.groupby('nearest_ivt_unit').describe().T
# Unpack index column so two-level index becomes two columns
results_stats_by_unit = results_stats_by_unit.reset_index()
rename_dict = {'level_0': 'variable', 'level_1': 'statistic'}
results_stats_by_unit = results_stats_by_unit.rename(columns=rename_dict)
# Save
results_stats_by_unit.to_csv(f'./output/results_stats_by_unit_{SCENARIO_NAME}.csv')
results_stats_by_unit.head()

## Map

In [ ]:
# Load the England and Wales LSOA shapefile
filename = "./data/Lower_layer_super_output_areas_(E+W)_2011_Boundaries_(Generalised_Clipped)_V2.zip"
lsoa_gdf = geopandas.read_file(filename)

# Set or convert to British National Grid (EPSG:27700)
if lsoa_gdf.crs is None:
    lsoa_gdf = lsoa_gdf.set_crs("EPSG:27700")
else:
    lsoa_gdf = lsoa_gdf.to_crs("EPSG:27700")

# Get the England and Wales outline
filename = "zip://./data/Countries_(December_2022)_GB_BFC.zip"
countries_gdf = geopandas.read_file(filename)
if countries_gdf.crs is None:
    countries_gdf = countries_gdf.set_crs("EPSG:27700")
else:
    countries_gdf = countries_gdf.to_crs("EPSG:27700")

mask = countries_gdf["CTRY22NM"].isin(["England", "Wales"])
countries_gdf = countries_gdf.loc[mask]
england = countries_gdf.boundary

# Load hospital coordinates from CSV and create a GeoDataFrame
hospitals_df = pd.read_csv("./data/stroke_hospitals_2025.csv")
hospitals_gdf = geopandas.GeoDataFrame(
    hospitals_df,
    geometry=geopandas.points_from_xy(
        pd.to_numeric(hospitals_df["long"]),
        pd.to_numeric(hospitals_df["lat"]),
    ),
    crs="EPSG:4326",
).to_crs("EPSG:27700")

hospitals_gdf = hospitals_gdf.loc[hospitals_gdf["Country"] == "England"]
hospitals_gdf.head()

# Change 'Use_MT' values for new MT centres to hospitals_gdf
if len(NEW_MT_CENTRES) > 0:
    for new_mt_centre in NEW_MT_CENTRES:
        hospitals_gdf.loc[hospitals_gdf['Postcode'] == new_mt_centre, 'Use_MT'] = 1

In [ ]:
# Merge the results with the LSOA shapefile
data = lsoa_gdf.merge(results, left_on='LSOA11NM', right_on='area_lower', how='right')

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

data['treated_improved_p_mrs_0-2'] = \
    data['drip_ship_mrs_0-2_improvement_over_no_treatment']

columns = ['nearest_ivt_time_no_bias',
           'nearest_mt_time',
           'treated_improved_p_mrs_0-2']

color_schemes = ['afmhot_r',
                 'gist_heat_r',
                 'inferno']

labels = ['Time to IVT unit (minutes)',
          'Time to MT unit (minutes, including transfer time)',
          'Additional probability of mRS 0-2, over untreated']

fig = plt.figure(figsize=(30, 10))

for i in range(3):
    ax = fig.add_subplot(1, 3, i + 1)

    column = columns[i]
    values = pd.to_numeric(data[column], errors='coerce').astype(float)
    valid_values = values.where(np.isfinite(values))

    vmin, vmax = valid_values.min(), valid_values.max()
    if vmin == vmax:
        vmax = vmin + 1

    cmap = plt.get_cmap(color_schemes[i])
    norm = Normalize(vmin=vmin, vmax=vmax)

    # Use a temporary numeric column; GeoPandas handles the color mapping.
    data['_plot_value'] = values

    data.plot(
        column='_plot_value',
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={'color': 'lightgrey'},
        antialiased=False,
        linewidth=0,
    )

    england.plot(ax=ax, edgecolor='k', facecolor='none', linewidth=1)

    ivt_mask = pd.to_numeric(hospitals_gdf['Use_IVT'], errors='coerce').eq(1)
    mt_mask = pd.to_numeric(hospitals_gdf['Use_MT'], errors='coerce').eq(1)

    hospitals_gdf.loc[ivt_mask].plot(
        ax=ax, edgecolor='k', facecolor='w', markersize=30, marker='o', zorder=99
    )
    hospitals_gdf.loc[mt_mask].plot(
        ax=ax, edgecolor='k', facecolor='w', markersize=200, marker='*', zorder=99
)

    fig.colorbar(
        ScalarMappable(norm=norm, cmap=cmap),
        ax=ax,
        shrink=0.5,      # shrink the colorbar to 50% of its original size
        pad=0.01,        # gap between map and colorbar (default 0.05)
        fraction=0.05,   # width of colorbar relative to the axes
        label=labels[i],
    )

    ax.set_title(labels[i], fontsize=16)

    ax.apply_aspect()
    ax.set_axis_off()
    
plt.savefig(f'./output/map_three_in_one_{SCENARIO_NAME}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

data['treated_improved_p_mrs_0-2'] = \
    data['lvo_drip_ship_ivt_mt_mrs_0-2'] - data['lvo_no_treatment_mrs_0-2']

columns = ['nearest_ivt_time_no_bias',
           'nearest_mt_time',
           'treated_improved_p_mrs_0-2']

color_schemes = ['afmhot_r',
                 'gist_heat_r',
                 'inferno']

labels = ['Time to IVT unit (minutes)',
          'Time to MT unit (minutes, including transfer time)',
          'Additional probability of mRS 0-2, over untreated (for LVO patients)']

fig = plt.figure(figsize=(30, 10))

for i in range(3):
    ax = fig.add_subplot(1, 3, i + 1)

    column = columns[i]
    values = pd.to_numeric(data[column], errors='coerce').astype(float)
    valid_values = values.where(np.isfinite(values))

    vmin, vmax = valid_values.min(), valid_values.max()
    if vmin == vmax:
        vmax = vmin + 1

    cmap = plt.get_cmap(color_schemes[i])
    norm = Normalize(vmin=vmin, vmax=vmax)

    # Use a temporary numeric column; GeoPandas handles the color mapping.
    data['_plot_value'] = values

    data.plot(
        column='_plot_value',
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={'color': 'lightgrey'},
        antialiased=False,
        linewidth=0,
    )

    england.plot(ax=ax, edgecolor='k', facecolor='none', linewidth=1)

    ivt_mask = pd.to_numeric(hospitals_gdf['Use_IVT'], errors='coerce').eq(1)
    mt_mask = pd.to_numeric(hospitals_gdf['Use_MT'], errors='coerce').eq(1)

    hospitals_gdf.loc[ivt_mask].plot(
        ax=ax, edgecolor='k', facecolor='w', markersize=30, marker='o', zorder=99
    )
    hospitals_gdf.loc[mt_mask].plot(
        ax=ax, edgecolor='k', facecolor='w', markersize=200, marker='*', zorder=99
)

    fig.colorbar(
        ScalarMappable(norm=norm, cmap=cmap),
        ax=ax,
        shrink=0.5,      # shrink the colorbar to 50% of its original size
        pad=0.01,        # gap between map and colorbar (default 0.05)
        fraction=0.05,   # width of colorbar relative to the axes
        label=labels[i],
    )

    ax.set_title(labels[i], fontsize=16)

    ax.apply_aspect()
    ax.set_axis_off()
    
plt.savefig(f'./output/map_three_in_one_{SCENARIO_NAME}_lvo.png', dpi=300, bbox_inches='tight')
plt.show()